In [ ]:
#Importing the correct libraries 
import pandas as pd
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt

# Load and clean data
df = pd.read_csv('/home/rt334/Downloads/MEDUSA_Master.csv')
df['Histology_clean'] = df['Histology'].str.strip().str.title()

age_acc_cols = [
    'ageAcc.Horvath', 'ageAcc.Hannum', 'ageAcc.Levine',
    'ageAcc.Hovarth2', 'ageAcc.PedBE', 'ageAcc.Wu',
    'ageAcc.TL', 'ageAcc.BLUP', 'ageAcc.EN'
]

# Cleaner display names for the report table
clock_display_names = {
    'ageAcc.Horvath': 'Horvath', 'ageAcc.Hannum': 'Hannum',
    'ageAcc.Levine': 'Levine (PhenoAge)', 'ageAcc.Hovarth2': 'Horvath (skin & blood)',
    'ageAcc.PedBE': 'PedBE', 'ageAcc.Wu': 'Wu', 'ageAcc.TL': 'Telomere Length (TL)',
    'ageAcc.BLUP': 'BLUP', 'ageAcc.EN': 'Elastic Net (EN)'
}

# Running the statistical tests
def run_group_tests(df, group_col, group_a, group_b, cols):
    results = []
    a = df[df[group_col] == group_a]
    b = df[df[group_col] == group_b]
    for col in cols:
        a_vals = a[col].dropna()
        b_vals = b[col].dropna()
        if len(a_vals) < 3 or len(b_vals) < 3:
            continue
        stat, p = mannwhitneyu(a_vals, b_vals)
        results.append({
            'Epigenetic clock': clock_display_names[col],
            f'n ({group_a})': len(a_vals), f'n ({group_b})': len(b_vals),
            f'Median ({group_a})': round(a_vals.median(), 2),
            f'Median ({group_b})': round(b_vals.median(), 2),
            'p-value': p
        })
    res_df = pd.DataFrame(results)
    if len(res_df) > 0:
        res_df['FDR'] = multipletests(res_df['p-value'], method='fdr_bh')[1]
        res_df['p-value'] = res_df['p-value'].round(3)
        res_df['FDR'] = res_df['FDR'].round(3)
    return res_df

hist_results = run_group_tests(df, 'Histology_clean', 'Epithelioid', 'Biphasic', age_acc_cols)
asb_results = run_group_tests(df, 'Asbestos exposure', 'Yes', 'No', age_acc_cols)

# Building a clean, report-ready table image 
def make_report_table(df, title, filename):
    fig, ax = plt.subplots(figsize=(12, 0.55 * len(df) + 1.6))
    ax.axis('off')

    col_labels = list(df.columns)
    cell_text = df.astype(str).values.tolist()

    table = ax.table(cellText=cell_text, colLabels=col_labels, cellLoc='center', loc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 1.8)

    # Header styling
    for col_idx in range(len(col_labels)):
        cell = table[0, col_idx]
        cell.set_facecolor('#2C3E50')
        cell.set_text_props(color='white', fontweight='bold')
        cell.set_edgecolor('white')

    # Body row styling — bold + green if FDR < 0.05 (significant)
    fdr_col_idx = col_labels.index('FDR')
    for row_idx in range(1, len(df) + 1):
        fdr_val = float(cell_text[row_idx - 1][fdr_col_idx])
        for col_idx in range(len(col_labels)):
            cell = table[row_idx, col_idx]
            cell.set_edgecolor('#DDDDDD')
            cell.set_facecolor('#F7F9FA' if row_idx % 2 == 0 else 'white')
            if fdr_val < 0.05:
                cell.set_text_props(fontweight='bold', color='#1B7837')

    plt.title(title, fontsize=12, fontweight='bold', pad=14, loc='left')
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"Saved {filename}")

# Generate both tables as pngs
make_report_table(hist_results,
    "Table X. Epigenetic age acceleration by histological subtype (Mann-Whitney U, FDR-corrected)",
    "table_histology.png")

make_report_table(asb_results,
    "Table Y. Epigenetic age acceleration by asbestos exposure status (Mann-Whitney U, FDR-corrected)",
    "table_asbestos.png")